[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

Implement **Layer Normalization** from scratch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last dimension**.

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `F.layer_norm` or `torch.nn.LayerNorm`
- Normalize over the last dimension only
- Must support autograd

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00


In [2]:
import torch

In [33]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True, unbiased=False)
    return gamma * (x - mean) / (std ** 2 + eps) ** 0.5 + beta


In [34]:
# 🧪 Debug
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
print(out)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print(ref)
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

tensor([[ 0.3751,  1.2160,  0.5409, -0.3298, -0.6542, -1.5960, -0.9808,  1.4289],
        [-0.7712,  0.4749, -0.8280,  1.3612, -1.2026,  1.3256,  0.6485, -1.0085]])
Your output mean: tensor([ 0.0000e+00, -4.4703e-08])
Your output std:  tensor([1.0690, 1.0690])
tensor([[ 0.3751,  1.2160,  0.5409, -0.3298, -0.6542, -1.5960, -0.9808,  1.4289],
        [-0.7712,  0.4749, -0.8280,  1.3612, -1.2026,  1.3256,  0.6485, -1.0085]])
Match ref?       True


In [26]:
# ✅ SUBMIT
from torch_judge import check
check("layernorm")


🧪 Testing: Implement LayerNorm (Medium)
──────────────────────────────────────────────────
torch.Size([2, 3, 1])
  ❌ [1/3] Shape and basic behavior
     Value mismatch vs F.layer_norm
torch.Size([4, 1])
  ❌ [2/3] With learned parameters
     Value mismatch with non-trivial gamma/beta
torch.Size([2, 1])
  ✅ [3/3] Gradient flow (1.0ms)
──────────────────────────────────────────────────
  📊 1/3 tests passed.
  Keep going! Use hint("layernorm") if you're stuck.

